# Gen 1 — simple use

Run **1–2**, then **3b** to load your saved calibration. On first use, run **3a** before **3b**.
Then choose **DC**, **triangle**, or **PID** and edit the values at the top of that example.
Run cells individually. **Run “Turn off” after connecting, before switching modes, and when finished.**
Closing the notebook does not disconnect outputs.

For **Z7010, profile 1 or 2**, OS 2.07+ within OS 2, using [`gen1-os2/main`](https://github.com/SinclairQuantumLab/pyrpl-wwlyn/tree/gen1-os2/main).


## 1. Check the version

Select the matching checkout's Python 3.14 kernel. This checks the branch and shows which PyRPL installation is loaded.


In [ ]:
import sys, subprocess, json
from pathlib import Path
from getpass import getpass
import time
import numpy as np
import pyrpl
from pyrpl import Pyrpl

package = Path(pyrpl.__file__).resolve().parent
branch = subprocess.check_output(["git", "-C", str(package), "branch", "--show-current"], text=True).strip()
assert sys.version_info[:2] == (3, 14), "Select a Python 3.14 kernel."
assert branch == 'gen1-os2/main', "Select the correct branch's kernel."
bitstream = package / 'fpga/red_pitaya.bin'
print("Python:", sys.version.split()[0], "| Branch:", branch)
print("PyRPL:", pyrpl.__file__)
print("FPGA image:", bitstream.name)


## 2. Connect

Edit `HOSTNAME`. This loads the board-specific FPGA image and starts PyRPL; leave the actuator disconnected during setup.
The loader checks the board/image compatibility before programming. Run once per kernel session.


In [ ]:
HOSTNAME = "rp-xxxxxx.local"  # Or the board's IP address.
calibration_file = Path(f"calibration_gen1_{HOSTNAME}.json")

p = Pyrpl(config="", hostname=HOSTNAME, password=getpass("SSH password: "),
          filename=str(bitstream), gui=False, reloadfpga=True, reloadserver=True)
rp = p.rp

print("Connected. OS:", rp.os_version)
profile = getattr(rp, "hardware_profile", {})
print("Board:", profile.get("zynq"), "| Profile:", profile.get("id"))


## 3. Calibration

### 3a. Measure and save — only for a new or changed setup

Connect **OUT1 → IN1** and measure that voltage with an external meter/scope.
Enter five measured voltages. Keep the input within its jumper-selected range.
The cell saves separate OUT1 and IN1 calibration coefficients to a JSON file in the working folder.
The filename includes this board type and hostname.

**For later sessions with the same setup, skip this sweep and run 3b.**
Recalibrate after changing the board, output load, input range or analog hardware.


In [ ]:
COMMANDS = [-0.30, -0.15, 0.0, 0.15, 0.30]  # Raw PyRPL units.

asg = rp.asg0
asg.setup(waveform="dc", amplitude=0, offset=0,
          trigger_source="immediately", output_direct="off")
out_raw, in_raw, measured_V = [], [], []
try:
    for command in COMMANDS:
        asg.offset = command
        asg.output_direct = "out1"
        time.sleep(0.3)
        voltage = float(input(f"Command {command:+.2f}: meter voltage [V] = "))
        reading = np.mean([rp.scope.voltage_in1 for _ in range(20)])
        assert np.isfinite(voltage) and abs(reading) < 0.98, "Check meter entry and IN1 clipping."
        out_raw.append(asg.offset)
        in_raw.append(reading)
        measured_V.append(voltage)
finally:
    asg.output_direct = "off"

assert np.ptp(in_raw) > 0.01 and np.ptp(measured_V) > 0.01, "Check loopback wiring and readings."
OUT_GAIN, OUT_OFFSET = np.polyfit(out_raw, measured_V, 1)
IN_GAIN, IN_OFFSET = np.polyfit(in_raw, measured_V, 1)
coefficients = dict(out_gain=float(OUT_GAIN), out_offset=float(OUT_OFFSET),
                    in_gain=float(IN_GAIN), in_offset=float(IN_OFFSET))
assert np.isfinite(list(coefficients.values())).all() and abs(OUT_GAIN) > 1e-6 and abs(IN_GAIN) > 1e-6
calibration_file.write_text(json.dumps(coefficients, indent=2) + "\n")
print("Saved calibration:", calibration_file.resolve())
print(coefficients)


### 3b. Load saved calibration and define the conversion functions

Run this every session, including after a new calibration. These are the four functions from your original notebook:
`rp_to_rigol`, `rigol_to_rp`, `rp_in1_to_rigol`, and `rigol_to_rp_in1`.
The DC, triangle, PID and readback cells below use them directly.

Keep the JSON file for future sessions. If you move the notebook to another folder/computer, copy that file too.


In [ ]:
if not calibration_file.exists():
    raise FileNotFoundError("Run 3a once, or copy this board's saved calibration JSON into this folder.")
cal = json.loads(calibration_file.read_text())
RP_OUT_TO_RIGOL_GAIN, RP_OUT_TO_RIGOL_OFFSET = cal["out_gain"], cal["out_offset"]
RP_IN1_TO_RIGOL_GAIN, RP_IN1_TO_RIGOL_OFFSET = cal["in_gain"], cal["in_offset"]

def rp_to_rigol(v_rp):
    return RP_OUT_TO_RIGOL_GAIN * v_rp + RP_OUT_TO_RIGOL_OFFSET

def rigol_to_rp(v_rigol):
    return (v_rigol - RP_OUT_TO_RIGOL_OFFSET) / RP_OUT_TO_RIGOL_GAIN

def rp_in1_to_rigol(v_rp_in1):
    return RP_IN1_TO_RIGOL_GAIN * v_rp_in1 + RP_IN1_TO_RIGOL_OFFSET

def rigol_to_rp_in1(v_rigol):
    return (v_rigol - RP_IN1_TO_RIGOL_OFFSET) / RP_IN1_TO_RIGOL_GAIN

print("Loaded calibration:", calibration_file.resolve())


## 4. DC voltage on OUT1

Edit `DC_V` and run. The output stays active until you run the **Turn off** cell or change the ASG settings.


In [ ]:
DC_V = 0.20

raw = rigol_to_rp(DC_V)
assert -0.95 <= raw <= 0.95, "Requested voltage is outside the output range."
rp.asg0.setup(waveform="dc", amplitude=0, offset=raw,
              trigger_source="immediately", output_direct="off")
rp.asg0.output_direct = "out1"


## 5. Triangle wave on OUT1

`PEAK_V` is half the peak-to-peak voltage. This fork calls a triangle `"ramp"`.


In [ ]:
FREQUENCY_HZ = 1000
PEAK_V = 0.20  # 0.40 V peak-to-peak.
CENTER_V = 0.0

offset = rigol_to_rp(CENTER_V)
amplitude = abs(rigol_to_rp(CENTER_V + PEAK_V) - offset)
assert FREQUENCY_HZ > 0 and PEAK_V > 0 and amplitude > 0 and abs(offset) + amplitude <= 0.95
rp.asg0.periodic = True
rp.asg0.setup(waveform="ramp", frequency=FREQUENCY_HZ, amplitude=amplitude,
              offset=offset, cycles_per_burst=0,
              trigger_source="immediately", output_direct="off")
rp.asg0.output_direct = "out1"


## 6a. PID with a software setpoint: IN1 → OUT1

**Use negative P and I values:** set `P_GAIN < 0` and `I_HZ < 0` for this positive-response setup. The FPGA error is measured input minus setpoint.
Start with **OUT1 → IN1 loopback**. For a real plant: sensor → IN1, OUT1 → actuator.
The error is **IN1 − setpoint**, so a positive-response plant normally uses negative gains.
These are small starting gains, not tuned values. `P_GAIN` is dimensionless; `I_HZ` is integral unity-gain frequency in Hz.
This driver supports **PI**; derivative is disabled. Set `I_HZ = 0` for P-only operation.

Set `MIN_OUTPUT_V` and `MAX_OUTPUT_V` in physical OUT1 volts. Both limits are converted through `rigol_to_rp()`, including output gain and offset.


In [ ]:
TARGET_V = 0.10
P_GAIN = -0.05
I_HZ = -10.0
MIN_OUTPUT_V = -0.30  # Physical OUT1 voltage, measured externally.
MAX_OUTPUT_V = 0.30

setpoint = rigol_to_rp_in1(TARGET_V)
assert abs(setpoint) <= 0.95
assert MIN_OUTPUT_V < MAX_OUTPUT_V, "Minimum output voltage must be below maximum."
min_raw, max_raw = sorted((rigol_to_rp(MIN_OUTPUT_V), rigol_to_rp(MAX_OUTPUT_V)))
assert -1 <= min_raw < max_raw <= 1 - 2**-13, "Calibrated limits exceed the FPGA output range."
pid = rp.pid0
pid.p = 0
pid.i = 0
pid.input = "in1"
pid.inputfilter = []
pid.differential_mode_enabled = False
pid.use_setpoint_sequence = False
pid.pause_gains = "off"  # Continuous operation; external hold disabled.
pid.paused = False
pid.min_voltage, pid.max_voltage = min_raw, max_raw
pid.setpoint = setpoint
pid.ival = 0
pid.p, pid.i = P_GAIN, I_HZ
pid.output_direct = "out1"


Rerun the software-setpoint PID cell to change target or gains. It resets the integrator.
If the output saturates or oscillates, run the **Turn off** cell, check the feedback sign and reduce gains.
This readback cell is for the software-setpoint PID0 mode; it does not stop the loop.
OUT1 estimate converts the internal controller signal; it is not an external voltage measurement.


In [ ]:
print("IN1 [V]:", rp_in1_to_rigol(rp.scope.voltage_in1))
print("Setpoint [V]:", rp_in1_to_rigol(rp.pid0.setpoint))
print("OUT1 estimate [V]:", rp_to_rigol(rp.pid0.current_output_signal))


## 6b. PID with IN2 as the setpoint

**Use negative P and I values:** set `P_GAIN < 0` and `I_HZ < 0` for this positive-response setup. The FPGA error is measured input minus setpoint.
Wire **sensor → IN1**, **external setpoint voltage → IN2**, and **OUT1 → actuator**.
Run **Turn off** before switching PID modes.

PID0 supplies the IN2 reference; PID1 drives OUT1 using **error = IN1 − IN2**.
Change the voltage on IN2 to change the target. The digital `pid.setpoint` is ignored in this mode.
Use matching IN1/IN2 input ranges and filters. The FPGA compares raw ADC signals;
Python calibration functions do not correct this subtraction. Input gain/offset mismatch needs separate compensation.

Set `MIN_OUTPUT_V` and `MAX_OUTPUT_V` in physical OUT1 volts. Both limits are converted through `rigol_to_rp()`, including output gain and offset.


In [ ]:
P_GAIN = -0.05
I_HZ = -10.0
MIN_OUTPUT_V = -0.30  # Physical OUT1 voltage, measured externally.
MAX_OUTPUT_V = 0.30
assert MIN_OUTPUT_V < MAX_OUTPUT_V, "Minimum output voltage must be below maximum."
min_raw, max_raw = sorted((rigol_to_rp(MIN_OUTPUT_V), rigol_to_rp(MAX_OUTPUT_V)))
assert -1 <= min_raw < max_raw <= 1 - 2**-13, "Calibrated limits exceed the FPGA output range."

# IN2 reference path: PID0 does not drive an output.
reference = rp.pid0
reference.output_direct = "off"
reference.p = reference.i = reference.ival = 0
reference.input = "in2"
reference.inputfilter = []
reference.differential_mode_enabled = False
reference.use_setpoint_sequence = False
reference.pause_gains = "off"
reference.paused = False

# Controller: PID1 compares IN1 with PID0's filtered IN2 input.
pid = rp.pid1
pid.p = 0
pid.i = 0
pid.input = "in1"
pid.inputfilter = []
pid.differential_mode_enabled = True
pid.use_setpoint_sequence = False
pid.pause_gains = "off"
pid.paused = False
pid.min_voltage, pid.max_voltage = min_raw, max_raw
pid.ival = 0
pid.p, pid.i = P_GAIN, I_HZ
pid.output_direct = "out1"


## 7. Turn off

This is the explicit module-by-module shutdown used in your original notebook.
It disconnects the output routes and prints each result. PID gains and integrator values are left unchanged.
A modified analog stage may still have a physical offset.


In [ ]:
for name in ("asg0", "asg1", "pid0", "pid1", "pid2", "iq0", "iq1", "iq2", "iir", "trig0", "trig1"):
    try:
        module = getattr(rp, name)
    except AttributeError:
        print(name, "not present")
        continue
    module.output_direct = "off"
    print(name, "-> off")


Offline checked; not tested on a connected board.
